# Federal Reserve Macro Insights MVP (Streamlined)

This notebook is the clean orchestrator version.
Core logic is now in `core/*.py` modules.

Original notebook preserved: `fed_macro_mvp.ipynb`.


## 0) Install / refresh dependencies
This section ensures the notebook runtime has every package required for scraping, indexing, retrieval, and local LLM generation. Run it whenever you create a new environment or after dependency changes, then restart the kernel if prompted.


In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 1) Imports and configuration
This section loads the pipeline modules and creates the central `cfg` object that controls paths, model names, retrieval parameters, and LLM settings. Treat this as the single source of truth for run behavior before any data processing starts.


In [2]:
from pathlib import Path
import pandas as pd

from core.pipeline import create_config, run_ingest_and_index, run_full_analysis, persist_results

cfg = create_config(Path.cwd())
print('Project dir:', cfg.project_dir)
print('Model:', cfg.ollama_model)
print('Profile:', cfg.profile_name)
print('Days back / Max PDFs:', cfg.days_back, '/', cfg.max_pdfs)
print('Allowed doc types:', cfg.allowed_doc_types)


/Users/atheeshkrishnan/AK/DEV/hawkdove/storied/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project dir: /Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/fed_macro_mvp
Model: llama3:8b
Profile: fast_default
Days back / Max PDFs: 180 / 24
Allowed doc types: ['fomc_minutes', 'mpr']


## 2) Optional runtime overrides
This section lets you quickly tune speed vs. quality without editing core code, such as context length, token budget, and reranker toggles. Use these overrides for experimentation, then move stable values into defaults later.


In [3]:
# One-line profile switch
cfg.set_profile('fast_default')  # change to 'full_default' for broader, slower runs

# Optional manual overrides
cfg.top_k_topic = 2
cfg.max_context_chars = 2400
cfg.ollama_num_predict = 380
cfg.enable_reranker = False

# Keep retries + robust parser enabled
cfg.ollama_max_retries = 3
cfg.retry_context_shrink = [1.0, 0.85, 0.7]
cfg.retry_predict_shrink = [1.0, 1.0, 0.9]
cfg.use_json_repair = True

print('Profile:', cfg.profile_name)
print('Days back / Max PDFs:', cfg.days_back, '/', cfg.max_pdfs)
print('Allowed doc types:', cfg.allowed_doc_types)
print('Overrides applied.')


Profile: fast_default
Days back / Max PDFs: 180 / 24
Allowed doc types: ['fomc_minutes', 'mpr']
Overrides applied.


## 3) Ingestion + indexing
This section discovers Federal Reserve PDFs, downloads missing files, parses text, chunks content, and builds retrieval indexes. The output is a local evidence base that later analysis steps query for topic-specific macro signals.


In [4]:
ingest_index_result = run_ingest_and_index(cfg)

catalog_df = ingest_index_result['catalog_df']
download_df = ingest_index_result['download_df']
chunks_df = ingest_index_result['chunks_df']
stage_metrics = ingest_index_result.get('stage_metrics', {})

print('Catalog candidates:', len(catalog_df))
print('Downloaded/available:', len(download_df))
print('Chunks:', len(chunks_df))
print('Stage metrics:', stage_metrics)
if stage_metrics.get('warnings'):
    for w in stage_metrics['warnings']:
        print('[warn]', w)

if not download_df.empty:
    cols = ['title', 'doc_type', 'date_hint', 'status', 'local_path']
    cols = [c for c in cols if c in download_df.columns]
    display(download_df[cols].head(10))


Batches: 100%|██████████| 6/6 [00:05<00:00,  1.16it/s]

Catalog candidates: 7
Downloaded/available: 7
Chunks: 171
Stage metrics: {'profile_name': 'fast_default', 'ingest_latency_s': 31.412973165512085, 'index_latency_s': 14.784824132919312, 'catalog_candidates': 7, 'downloaded_or_exists': 7, 'chunks': 171, 'warnings': []}


,title,doc_type,date_hint,status,local_path
0,PDF,mpr,2026-03-18,downloaded,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
1,PDF,mpr,2026-01-28,downloaded,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
2,PDF,fomc_minutes,2026-01-28,downloaded,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
3,PDF,mpr,2025-12-10,downloaded,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
4,PDF,fomc_minutes,2025-12-10,downloaded,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
5,PDF,mpr,2025-10-29,downloaded,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
6,PDF,fomc_minutes,2025-10-29,downloaded,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...


## 4) Macro analysis
This section runs hybrid retrieval by macro topic, constructs context, generates a structured investor view with the local model, and validates the output schema. It also prints normalized JSON and citation quotes so each claim is tied to readable evidence.


In [5]:
analysis_result = run_full_analysis(cfg)

print('Sparse enabled:', analysis_result['sparse_enabled'])
print('Reranker enabled:', analysis_result['reranker_enabled'])
print('Timings:', analysis_result['timings'])
if 'analysis_counts' in analysis_result:
    print('Analysis counts:', analysis_result['analysis_counts'])
if analysis_result.get('analysis_warnings'):
    for w in analysis_result['analysis_warnings']:
        print('[warn]', w)

display(analysis_result['topic_summary_df'])
if not analysis_result['hits_df'].empty:
    display(analysis_result['hits_df'][['topic', 'chunk_id', 'doc_id', 'final_score', 'recency']].head(12))
if not analysis_result['attempt_log'].empty:
    display(analysis_result['attempt_log'])

if 'citation_preview_df' in analysis_result and not analysis_result['citation_preview_df'].empty:
    display(analysis_result['citation_preview_df'][['doc_id', 'chunk_id', 'quote']].head(8))

print(analysis_result.get('normalized_json_text', analysis_result['llm_text']))
if analysis_result['parsed'] is None:
    print('[check] JSON parse failed after retries')
else:
    quality = analysis_result['quality']
    print('[check] Missing top keys:', quality['missing_top_keys'] if quality['missing_top_keys'] else 'None')
    print('[check] Bad shape:', quality['bad_shape'] if quality['bad_shape'] else 'None')
    print('[check] Bad evidence IDs:', quality['bad_evidence_ids'] if quality['bad_evidence_ids'] else 'None')
    print('[check] Unknown citation IDs:', quality['unknown_citation_ids'] if quality['unknown_citation_ids'] else 'None')
    print('[check] Missing topics:', quality['missing_topics'] if quality['missing_topics'] else 'None')
    print('[check] Extra topics:', quality['extra_topics'] if quality['extra_topics'] else 'None')


Sparse enabled: True
Reranker enabled: False
Timings: {'sparse_latency_s': 0.04918980598449707, 'rerank_load_s': 2.3126602172851562e-05, 'retrieval_latency_s': 0.7592079639434814, 'llm_stage_s': 120.23660111427307}
Analysis counts: {'index_docs': 7, 'index_chunks': 171, 'retrieved_unique_chunks': 12}


,topic,hits,top_final_score,top_chunk_id
0,inflation,2,0.889270,fomcminutes20251210.pdf::chunk0022
1,unemployment,2,0.880409,fomcminutes20251210.pdf::chunk0014
2,growth,2,0.938959,fomcminutes20260128.pdf::chunk0040
3,policy_rates,2,0.853539,monetary20251029a1.pdf::chunk0002
4,financial_conditions,2,0.853539,fomcminutes20251029.pdf::chunk0002
5,credit,2,0.938959,fomcminutes20260128.pdf::chunk0015


,topic,chunk_id,doc_id,final_score,recency
0,growth,fomcminutes20260128.pdf::chunk0040,fomcminutes20260128.pdf,0.938959,0.825596
1,credit,fomcminutes20260128.pdf::chunk0015,fomcminutes20260128.pdf,0.938959,0.825596
2,inflation,fomcminutes20251210.pdf::chunk0022,fomcminutes20251210.pdf,0.889270,0.683629
3,unemployment,fomcminutes20251210.pdf::chunk0014,fomcminutes20251210.pdf,0.880409,0.683629
4,unemployment,fomcminutes20251210.pdf::chunk0027,fomcminutes20251210.pdf,0.876610,0.683629
5,policy_rates,monetary20251029a1.pdf::chunk0002,monetary20251029a1.pdf,0.853539,0.581541
6,financial_conditions,fomcminutes20251029.pdf::chunk0002,fomcminutes20251029.pdf,0.853539,0.581541
7,financial_conditions,fomcminutes20251029.pdf::chunk0016,fomcminutes20251029.pdf,0.819479,0.581541
8,policy_rates,monetary20251210a1.pdf::chunk0002,monetary20251210a1.pdf,0.800116,0.683629
9,credit,fomcminutes20251210.pdf::chunk0019,fomcminutes20251210.pdf,0.759326,0.683629


,attempt,status,latency_s,context_chars,num_predict
0,1,ok,120.14,2400,380


,doc_id,chunk_id,quote
0,fomcminutes20251210.pdf,fomcminutes20251210.pdf::chunk0022,2027 and 2028. Tariff increases were expected ...
1,fomcminutes20251210.pdf,fomcminutes20251210.pdf::chunk0034,"the target range for the federal funds rate, s..."
2,fomcminutes20251210.pdf,fomcminutes20251210.pdf::chunk0014,ecessary to implement the Committee’s framewor...
3,fomcminutes20251210.pdf,fomcminutes20251210.pdf::chunk0027,ekly initial unemployment insurance claims and...
4,fomcminutes20260128.pdf,fomcminutes20260128.pdf::chunk0040,the discussion of developments in financial ma...
5,fomcminutes20251210.pdf,fomcminutes20251210.pdf::chunk0016,e available indicators suggested that real GDP...
6,monetary20251029a1.pdf,monetary20251029a1.pdf::chunk0002,Philip N. Jefferson; Alberto G. Musalem; and C...
7,monetary20251210a1.pdf,monetary20251210a1.pdf::chunk0002,"air; John C. Williams, Vice Chair; Michael S. ..."


{
  "generated_at_utc": "2026-03-18T18:28:34Z",
  "executive_summary": "Recent Federal Reserve communications suggest a mixed macro view for the US economy.",
  "regime_call": {
    "growth_momentum": "Moderate",
    "inflation_trend": "Upswing",
    "policy_bias": "Neutral",
    "recession_risk": "Low",
    "confidence": "0.85"
  },
  "topic_signals": [
    {
      "topic": "inflation",
      "view": "No clear signal",
      "confidence": "medium",
      "evidence": [
        "fomcminutes20251210.pdf::chunk0022",
        "fomcminutes20251210.pdf::chunk0034"
      ]
    },
    {
      "topic": "unemployment",
      "view": "No clear signal",
      "confidence": "medium",
      "evidence": [
        "fomcminutes20251210.pdf::chunk0014",
        "fomcminutes20251210.pdf::chunk0027"
      ]
    },
    {
      "topic": "growth",
      "view": "No clear signal",
      "confidence": "medium",
      "evidence": [
        "fomcminutes20260128.pdf::chunk0040",
        "fomcminutes20251210.pdf::

## 5) Save artifacts
This section writes run outputs (retrieval hits, text response, and parsed JSON) to timestamped files for auditability and comparison across runs. Use these artifacts to track model behavior changes over time.


In [6]:
saved_paths = persist_results(cfg, analysis_result)
print(saved_paths)

{'hits_path': PosixPath('/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/fed_macro_mvp/outputs/hits_20260318_142834.csv'), 'text_path': PosixPath('/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/fed_macro_mvp/outputs/macro_answer_20260318_142834.txt'), 'json_path': PosixPath('/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/fed_macro_mvp/outputs/macro_answer_20260318_142834.json')}
